# OSM Business Finder by Region

Collects a list of businesses (name, category, address, website and public business phone) for a chosen country or state/province using the free **OpenStreetMap Overpass API**. It replaces several near-identical notebooks I made for Canada, Australia, the US, India, South Africa and Europe; you now just pick a region code. All data comes from OpenStreetMap (© OpenStreetMap contributors, ODbL licence).

**How to run**
1. Open in Google Colab and run the cells from top to bottom.
2. Change `REGION` and `CATEGORIES` in the settings cell.
3. The result is saved as a CSV file in the Colab file panel (download it from there).

**Responsible use**: only public business-level fields are collected, no personal data or emails. Keep queries small, respect the [Overpass usage policy](https://wiki.openstreetmap.org/wiki/Overpass_API#Public_Overpass_API_instances) and credit OpenStreetMap when you reuse the data. Do not use the output for unsolicited bulk marketing.

_Rewritten from my original Colab notebooks: `Canada lead business`, `free-osm-lead-finder-australia`, `us states`, `new lead colection v2`, `India business crawler v2.0 Place portal`._

> Note: the India notebook originally used a third-party commercial places service; this version collects the same kind of data from OpenStreetMap instead.

## 1. Settings

`REGION` is an ISO 3166 code: a country (`DE`, `NL`, `ZA`) or a state/province (`CA-ON`, `AU-NSW`, `US-TX`, `IN-WB`). Start with a small region and one or two categories.

In [ ]:
REGION = "CA-PE"          # e.g. "CA-ON", "AU-VIC", "US-ME", "IN-WB", "DE", "NL"
CATEGORIES = {            # OSM key -> list of values
    "shop": ["bakery", "hardware"],
    "amenity": ["restaurant"],
}
TIMEOUT_SECONDS = 180
PAUSE_BETWEEN_QUERIES = 10   # seconds, be polite to the free server

## 2. Helper: call the Overpass API with retries

In [ ]:
import time
import requests
import pandas as pd

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "osm-practice-notebook/1.0 (educational use)"}

def run_overpass(query, retries=4, wait=30):
    """Send a query to Overpass. On errors / rate limits, wait longer each time and retry."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=200)
            if r.status_code == 200:
                return r.json()["elements"]
            print(f"Attempt {attempt}: HTTP {r.status_code}")
        except requests.RequestException as e:
            print(f"Attempt {attempt}: {e}")
        time.sleep(wait * attempt)  # simple backoff: 30s, 60s, 90s ...
    raise RuntimeError("Overpass request failed after all retries")

## 3. Build the Overpass query

States/provinces use the `ISO3166-2` tag, countries use `ISO3166-1`. `nwr` means nodes, ways and relations; `out center tags` gives one point per place plus its tags.

In [ ]:
def build_query(region, key, value):
    area_tag = "ISO3166-2" if "-" in region else "ISO3166-1"
    return f"""
[out:json][timeout:{TIMEOUT_SECONDS}];
area["{area_tag}"="{region}"]["boundary"="administrative"]->.a;
nwr["{key}"="{value}"]["name"](area.a);
out center tags;
"""

print(build_query(REGION, "shop", "bakery"))

## 4. Convert OSM elements to table rows

In [ ]:
def element_to_row(el):
    """Keep only public, business-level fields from the OSM tags."""
    t = el.get("tags", {})
    address = " ".join(x for x in [t.get("addr:housenumber", ""), t.get("addr:street", "")] if x)
    return {
        "name": t.get("name", ""),
        "category": t.get("shop") or t.get("amenity") or t.get("office") or t.get("craft") or t.get("tourism", ""),
        "address": address,
        "city": t.get("addr:city", ""),
        "postcode": t.get("addr:postcode", ""),
        "website": t.get("website") or t.get("contact:website", ""),
        "phone": t.get("phone") or t.get("contact:phone", ""),  # public business phone, if mapped in OSM
        "osm_id": f"{el['type']}/{el['id']}",
    }

## 5. Run the queries

In [ ]:
rows = []
for key, values in CATEGORIES.items():
    for value in values:
        print(f"Fetching {key}={value} in {REGION} ...")
        elements = run_overpass(build_query(REGION, key, value))
        rows.extend(element_to_row(el) for el in elements)
        print(f"  {len(elements)} places")
        time.sleep(PAUSE_BETWEEN_QUERIES)

df = pd.DataFrame(rows).drop_duplicates(subset="osm_id")
print("Total places:", len(df))
df.head()

## 6. Quick summary and save to CSV

In [ ]:
print(df["category"].value_counts())
print("With website:", (df["website"] != "").sum())

filename = f"osm_businesses_{REGION}.csv"
df.to_csv(filename, index=False)
print("Saved", filename)